# Quantization

Store a float tensor as small integers plus a float scale, and reconstruct on use:

$$w \approx s \cdot q, \qquad q \in [-Q, Q] \subset \text{int8}, \qquad Q = 127$$

Symmetric (no zero point), so the scale is set by the largest magnitude in the group:

$$s = \frac{\max |w|}{Q}, \qquad q = \text{round}\!\left(\frac{w}{s}\right)$$

Rounding to the nearest of $2Q+1$ levels bounds the reconstruction error at half a step:

$$|w - s \cdot q| \le \frac{s}{2}$$

The **group** is the whole choice. `dim=-1` scales per row of the weight matrix
(per-channel); `dim=None` scales the whole matrix at once (per-tensor). Under one
shared scale a quiet row gets only $Q \cdot \frac{\max|w_{\text{row}}|}{\max|w|}$
levels — the rest of the range is reserved for a magnitude it never reaches.

Implementation and its fast tests: `src/video/quantize.py`. This notebook holds the
sweeps too slow to run on every test.

In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.append(str(root / "src"))
sys.path.append(str(root / "src" / "video"))

import torch

from checkpoint import load_checkpoint
from dataset import BinDataset
from evaluate import bits_per_char, full_loss
from paths import CKPT_DIR
from quantize import nbytes, quantize_model

CKPT = CKPT_DIR / "big_2026-08-30_09-09-16.pt"
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def load():
    """A fresh fp32 copy of the trained 27M checkpoint, in eval mode."""
    m, _ = load_checkpoint(CKPT, DEV)
    return m.eval()


def kl_vs(ref_logprobs, model, x):
    """KL(fp32 || quantized) over the same batch, in nats per token."""
    with torch.no_grad():
        lp = model(x).float().log_softmax(-1)
    return (ref_logprobs.exp() * (ref_logprobs - lp)).sum(-1).mean().item()


def val_batch(n, block_size):
    val = BinDataset("val")
    x = torch.stack([val.tokens(i * block_size, block_size) for i in range(n)])
    val.close()
    return x.to(DEV)

## Probe 1 — what does int8 actually cost?

Deterministic sweep over 2000 val windows (~1.02M tokens), so the deltas are exact
rather than sampled. **~27 s.**

Recorded result:

| variant | bpc | Δ bpc | KL nats | MB | shrink |
|---|---|---|---|---|---|
| fp32 baseline | 0.4985 | — | — | 104.0 | 1.00x |
| per-channel, all | 0.4986 | +0.0001 | 1.78e-04 | 26.2 | 3.97x |
| per-tensor, all | 0.4990 | +0.0005 | 1.08e-03 | 26.0 | 3.99x |
| per-channel, skip embeds | 0.4986 | +0.0001 | 1.41e-04 | 33.0 | 3.16x |

Three things it settles. int8 weight-only is **close to free** — 4x smaller for
0.02% of the quality. **KL separates what bpc cannot**: per-tensor is 5x worse in
bpc's fourth decimal but a clean **6.1x** in KL, at a fraction of the compute.
And **quantize the embeddings** — skipping them costs 3.97x → 3.16x to save
0.4e-04 nats, which is not a trade worth making.

In [3]:
MAX_WINDOWS = 2000

base = load()
bs = base.block_size
x = val_batch(8, bs)
with torch.no_grad():
    ref = base(x).float().log_softmax(-1)

val = BinDataset("val")
base_loss = full_loss(base, val, 16, bs, DEV, MAX_WINDOWS)
base_mb = nbytes(base.state_dict()) / 2**20
base_bpc = bits_per_char(base_loss)

VARIANTS = {
    "per-channel, all": dict(dim=-1),
    "per-tensor,  all": dict(dim=None),
    "per-channel, skip embeds": dict(
        dim=-1, skip=("token_embedding_table", "position_embedding_table", "lm_head")
    ),
}

print(f"{'variant':<26} {'bpc':>8} {'d bpc':>9} {'KL nats':>10} {'MB':>7} {'shrink':>8}")
print(f"{'fp32 baseline':<26} {base_bpc:>8.4f} {'--':>9} {'--':>10} {base_mb:>7.1f} {'1.00x':>8}")

for name, kw in VARIANTS.items():
    m = quantize_model(load(), **kw)
    kl = kl_vs(ref, m, x)
    bpc = bits_per_char(full_loss(m, val, 16, bs, DEV, MAX_WINDOWS))
    mb = nbytes(m.state_dict()) / 2**20
    print(f"{name:<26} {bpc:>8.4f} {bpc - base_bpc:>+9.4f} {kl:>10.2e} {mb:>7.1f} {base_mb / mb:>7.2f}x")
    del m
    torch.cuda.empty_cache()

val.close()

variant                         bpc     d bpc    KL nats      MB   shrink
fp32 baseline                0.4985        --         --   104.0    1.00x
per-channel, all             0.4986   +0.0001   1.78e-04    26.2    3.97x
per-tensor,  all             0.4990   +0.0005   1.08e-03    26.0    3.99x
per-channel, skip embeds     0.4986   +0.0001   1.41e-04    33.0    3.16x


## Probe 2 — where per-channel earns its keep

`quantize.py` samples four weights; this is every 2D weight in the model, sorted by
how badly a shared scale would treat its quietest row. **< 1 s.**

Recorded result: **worst 12.3x, median 3.5x, best 1.8x.**

| weight | ratio | per-tensor gives |
|---|---|---|
| `position_embedding_table.weight` | 12.3x | 10 of 127 |
| `blocks.{0,1}.attn.proj.weight` | 8.1x | 16 of 127 |
| `blocks.1.attn.qkv.weight` | 6.9x | 18 of 127 |
| `token_embedding_table.weight` = `lm_head.weight` | 2.7x | 47 of 127 |
| `blocks.6.ffwd.down.weight` | 1.8x | 70 of 127 |

Three readings. The **position table is the worst in the model** by a wide margin —
early positions are seen in every window and late ones only in long ones, so their
magnitudes diverge; a single scale would leave its quietest row 10 of 127 levels.
**Attention beats FFN for spread**, and **early blocks beat late ones**, monotonically
enough to look structural rather than incidental. And the tied pair appears twice with
identical ratios, which is the sharing from `quantize_model` showing up in the data.

Real outlier *features* — the 100x+ magnitudes that make per-tensor int8 fail outright
— only appear past roughly 6.7B parameters. At 27M this is a preview, not the full
version.

In [2]:
sd = torch.load(CKPT, map_location="cpu", weights_only=False)["model"]
QMAX = 127

rows = []
for name, w in sd.items():
    if w.ndim != 2:
        continue
    rm = w.float().abs().amax(-1)
    rows.append((( rm.max() / rm.min()).item(), name, w.shape[0], (QMAX * rm.min() / rm.max()).item()))

rows.sort(reverse=True)
print(f"{'weight':<34} {'rows':>5} {'ratio':>7} {'per-tensor gives':>18}")
for ratio, name, nrows, levels in rows:
    print(f"{name:<34} {nrows:>5} {ratio:>6.1f}x {f'{levels:.0f} of {QMAX}':>18}")

print(f"\nworst {rows[0][0]:.1f}x   median {sorted(r[0] for r in rows)[len(rows) // 2]:.1f}x   best {rows[-1][0]:.1f}x")

weight                              rows   ratio   per-tensor gives
position_embedding_table.weight      512   12.3x          10 of 127
blocks.1.attn.proj.weight            512    8.1x          16 of 127
blocks.0.attn.proj.weight            512    8.1x          16 of 127
blocks.1.attn.qkv.weight            1536    6.9x          18 of 127
blocks.2.attn.qkv.weight            1536    6.5x          20 of 127
blocks.3.attn.qkv.weight            1536    6.0x          21 of 127
blocks.6.attn.qkv.weight            1536    5.9x          21 of 127
blocks.4.attn.qkv.weight            1536    5.7x          22 of 127
blocks.0.attn.qkv.weight            1536    5.4x          23 of 127
blocks.7.attn.qkv.weight            1536    5.1x          25 of 127
blocks.5.attn.qkv.weight            1536    4.8x          26 of 127
blocks.7.attn.proj.weight            512    4.4x          29 of 127
blocks.1.ffwd.down.weight            512    4.3x          30 of 127
blocks.3.attn.proj.weight            512    3.9x